In [31]:
from pathlib import Path
import sys
from sklearn.model_selection import KFold

import numpy as np
import pandas as pd

%load_ext autoreload
%autoreload 2

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.data import load_train_data
from src.preprocessing import identify_column_types
from src.preprocessing import make_preprocessor
from src.models import make_ridge_pipeline
from src.evaluation import evaluate_model

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [32]:
train = load_train_data("../data/raw/train.csv")

X = train.drop(columns=["Id", "SalePrice"])
y = train["SalePrice"]

In [33]:
numeric_columns, categorical_columns = identify_column_types(X)

In [34]:
print(type(numeric_columns))
print(type(categorical_columns))

print(numeric_columns[:5])
print(categorical_columns[:5])

<class 'list'>
<class 'list'>
['MSSubClass', 'LotFrontage', 'LotArea', 'OverallQual', 'OverallCond']
['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour']


In [35]:
preprocessor = make_preprocessor(numeric_columns, categorical_columns)

In [36]:
model = make_ridge_pipeline(preprocessor, alpha = 10.0)

In [37]:
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

In [38]:
results = evaluate_model(model, X, y, cv)

In [39]:
results

{'train_rmse_mean': np.float64(0.10871475538254807),
 'train_rmse_std': np.float64(0.0064624374983536965),
 'valid_rmse_mean': np.float64(0.146784829890847),
 'valid_rmse_std': np.float64(0.039320124284981245),
 'train_rmse': array([0.11043961, 0.11161096, 0.0960188 , 0.11139886, 0.11410555]),
 'valid_rmse': array([0.13609998, 0.12937465, 0.22439085, 0.12771776, 0.11634091])}

In [41]:
path = Path("../results/experiments.csv")

new_row = pd.DataFrame([{
    "model": "ridge",
    "metric": "rmse_log",
    "valid_rmse": results["valid_rmse_mean"],
    "notes": "alpha=10",
}])

new_row.to_csv(path, mode="a", header = not path.exists(), index = False)